# Kiểm thử Kimi Vision với hóa đơn điện tử

Notebook đọc cấu hình từ `.env` tại thư mục gốc `InvoiceReferee` và gửi ảnh hóa đơn dưới dạng data URL.

In [7]:
import base64
import json
import mimetypes
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI


def find_project_root(start: Path) -> Path:
    """Tìm thư mục InvoiceReferee từ vị trí chạy notebook hiện tại."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".env").is_file() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Không tìm thấy thư mục gốc InvoiceReferee chứa cả .env và data/. "
        "Hãy mở notebook từ bên trong dự án."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
ENV_PATH = PROJECT_ROOT / ".env"
IMAGE_PATH = PROJECT_ROOT / "data" / "image" / "Hoa_don_dien_tu.jpg"

load_dotenv(ENV_PATH, override=True)

token = os.getenv("KIMI_TOKEN", "").strip()
secret = os.getenv("KIMI_SECRET", "").strip()
base_url = os.getenv("KIMI_BASE_URL", "").strip()
model = os.getenv("KIMI_MODEL", "").strip()

missing = [
    name
    for name, value in {
        "KIMI_TOKEN": token,
        "KIMI_SECRET": secret,
        "KIMI_BASE_URL": base_url,
        "KIMI_MODEL": model,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(f"Thiếu cấu hình trong {ENV_PATH}: {', '.join(missing)}")
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy ảnh: {IMAGE_PATH}")

print("Thư mục dự án:", PROJECT_ROOT)
print("Tệp cấu hình:", ENV_PATH)
print("Ảnh đầu vào:", IMAGE_PATH)
print("Model:", model)

Thư mục dự án: D:\NguyenHoangHa_nam4\MLAI\Accounting Agent\code\document\InvoiceReferee
Tệp cấu hình: D:\NguyenHoangHa_nam4\MLAI\Accounting Agent\code\document\InvoiceReferee\.env
Ảnh đầu vào: D:\NguyenHoangHa_nam4\MLAI\Accounting Agent\code\document\InvoiceReferee\data\image\Hoa_don_dien_tu.jpg
Model: moonshotai/Kimi-K3


In [8]:
mime_type = mimetypes.guess_type(IMAGE_PATH.name)[0] or "image/jpeg"
image_base64 = base64.b64encode(IMAGE_PATH.read_bytes()).decode("ascii")
image_data_url = f"data:{mime_type};base64,{image_base64}"

client = OpenAI(
    base_url=base_url,
    api_key=f"{token}.{secret}",
)

print(f"Đã mã hóa ảnh {IMAGE_PATH.name}: {len(image_base64):,} ký tự base64")

Đã mã hóa ảnh Hoa_don_dien_tu.jpg: 237,600 ký tự base64


In [9]:
completion = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": (
                "Bạn là tác tử kế toán chuyên đọc hóa đơn Việt Nam. "
                "Chỉ trích xuất thông tin nhìn thấy trong ảnh, không tự suy đoán."
            ),
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Đọc hóa đơn này và trả về JSON hợp lệ gồm: "
                        "seller_name, seller_tax_code, buyer_name, buyer_tax_code, "
                        "invoice_date, template_number, serial_number, invoice_number, "
                        "items, total_amount, amount_in_words và extraction_warnings. "
                        "Trường nào không đọc được thì dùng null. Không bọc JSON trong Markdown."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {"url": image_data_url},
                },
            ],
        },
    ],
    temperature=0.1,
    max_tokens=4096,
    top_p=0.95,
    stream=False,
)

response_text = completion.choices[0].message.content
print(response_text)

{
  "seller_name": "CÔNG TY KẾ TOÁN THIÊN ƯNG",
  "seller_tax_code": "0110329220",
  "buyer_name": "CÔNG TY CỔ PHẦN KIẾN TRÚC VÀ XÂY DỰNG IGS",
  "buyer_tax_code": "0110329573",
  "invoice_date": "11/07/2023",
  "template_number": "2C23TTU",
  "serial_number": "2C23TTU",
  "invoice_number": "00002438",
  "items": [
    {
      "no": 1,
      "description": "Khóa học thực hành kế toán tổng hợp",
      "unit": "Khóa",
      "quantity": 2,
      "unit_price": 3500000,
      "amount": 7000000
    },
    {
      "no": 2,
      "description": "Dịch vụ kế toán thuế quý 3/2023",
      "unit": "Quý",
      "quantity": 1,
      "unit_price": 2000000,
      "amount": 2000000
    }
  ],
  "total_amount": 9000000,
  "amount_in_words": "Chín triệu đồng./.",
  "extraction_warnings": [
    "Trường 'Mẫu số - Ký hiệu' trên hóa đơn hiển thị gộp một giá trị duy nhất '2C23TTU', nên cả template_number và serial_number đều ghi nhận cùng giá trị này.",
    "Hóa đơn ghi cả tên người mua cá nhân 'Nguyễn Thị Mai

In [10]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "output" / "Hoa_don_dien_tu.kimi.json"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

try:
    parsed_output = json.loads(response_text)
except json.JSONDecodeError as error:
    raise ValueError("Model không trả về JSON hợp lệ. Hãy kiểm tra response_text.") from error

with OUTPUT_PATH.open("w", encoding="utf-8") as output_file:
    json.dump(parsed_output, output_file, ensure_ascii=False, indent=2)

print("Đã lưu JSON:", OUTPUT_PATH)

Đã lưu JSON: D:\NguyenHoangHa_nam4\MLAI\Accounting Agent\code\document\InvoiceReferee\data\output\Hoa_don_dien_tu.kimi.json
